1. ENTSO-E Energy Data Analysis:

Retrieves electricity market data from the ENTSO-E Transparency Platform via entsoe-py for 8 European bidding zones — AT, BE, BG, CZ, FR, GR, HU, ES — covering April to November 2024 (UTC).
Authenticated via transparency.entsoe.eu (https://transparency.entsoe.eu/) API key. NoMatchingDataError is imported to handle missing data for certain country/period combinations.

In [2]:
from entsoe import EntsoePandasClient
from entsoe.exceptions import NoMatchingDataError
import pandas as pd

client = EntsoePandasClient(api_key="7308a46a-2e78-4bf2-a552-3295649e49e3")

countries = ['AT','BE','BG','CZ','FR','GR','HU','ES']

start = pd.Timestamp("2024-04-01", tz="UTC")
end   = pd.Timestamp("2024-11-01", tz="UTC")

2. Helper to get hourly solar production and solar price data for one country:

Queries the ENTSO-E API for a given country and period, returning a clean hourly data.
Key handling steps: Resampling for both solar power production and prices, and an inner join to retain only timestamps where both series are available. Returns None on any missing or empty response.

In [3]:
def get_hourly_solar_and_price(client, country_code, start, end):
    df_gen = client.query_generation(country_code, start=start, end=end)
    if df_gen.empty:
        return None

    if isinstance(df_gen.columns, pd.MultiIndex):
        mask = df_gen.columns.get_level_values(1) == "Actual Aggregated"
        df_agg = df_gen.loc[:, mask].copy()
        df_agg.columns = df_agg.columns.get_level_values(0)
    else:
        df_agg = df_gen.copy()

    solar = df_agg.get("Solar")
    if solar is None:
        return None

    solar_hourly = solar.resample("1h").mean()

    price_code = country_code

    try:
        prices = client.query_day_ahead_prices(
            country_code=price_code, start=start, end=end
        )
    except NoMatchingDataError:
        return None
    except Exception:
        return None

    if prices.empty:
        return None

    prices_hourly = prices.resample("1h").mean()

    df = pd.DataFrame(index=solar_hourly.index)
    df["solar_MW"] = solar_hourly
    df = df.join(prices_hourly.to_frame(name="price_EUR_MWh"), how="inner")
    df = df.dropna()
    if df.empty:
        return None

    df["hour"] = df.index.hour
    return df


3. Data Fetching & Strong Solar Hours Classification:

For each country, hourly solar generation and day-ahead prices are fetched and classified. Hours where the mean solar output exceeds 20% of the daily peak average are flagged as strong solar hours — used downstream to analyze the solar-price correlation. Failed or empty responses are collected in failed for transparency.

In [4]:
country_dfs = {}
strong_hours_by_country = {}
failed = []

for cc in countries:
    print(f"Fetching solar & prices for {cc} ...", end=" ")
    df = get_hourly_solar_and_price(client, cc, start, end)
    if df is None or df.empty:
        print("failed or no data")
        failed.append(cc)
        continue

    hourly_profile = df.groupby("hour")["solar_MW"].mean()
    max_avg = hourly_profile.max()
    thr_strong = 0.20 * max_avg

    strong_hours = hourly_profile[hourly_profile > thr_strong].index.tolist()
    strong_hours_by_country[cc] = strong_hours

    df["is_strong_solar_hour"] = df["hour"].isin(strong_hours)

    country_dfs[cc] = df

    print(f"ok | strong solar hours: {strong_hours}")

print("\nCountries with missing/failed data:", failed)

Fetching solar & prices for AT ... ok | strong solar hours: [8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]
Fetching solar & prices for BE ... ok | strong solar hours: [8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]
Fetching solar & prices for BG ... ok | strong solar hours: [8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]
Fetching solar & prices for CZ ... ok | strong solar hours: [8, 9, 10, 11, 12, 13, 14, 15, 16, 17]
Fetching solar & prices for FR ... ok | strong solar hours: [8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]
Fetching solar & prices for GR ... ok | strong solar hours: [8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]
Fetching solar & prices for HU ... ok | strong solar hours: [7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]
Fetching solar & prices for ES ... ok | strong solar hours: [9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]

Countries with missing/failed data: []


4. Hypothesis test: Solar Hours Prices vs. the Rest Hours Prices:

For each country, hourly data is split into strong solar vs. remaining hours to compare mean and median day-ahead prices across the two groups. A Pearson correlation coefficient and p-value between solar MW and price EUR MWh are computed per country — a negative correlation with p < 0.05 indicates a statistically significant merit-order effect (higher solar suppressing prices).

In [5]:
from scipy.stats import pearsonr

# Summary statistics
rows = []

for cc, df in country_dfs.items():
    strong = df[df["is_strong_solar_hour"]]
    rest = df[~df["is_strong_solar_hour"]]

    strong_mean_price = strong["price_EUR_MWh"].mean()
    strong_median_price = strong["price_EUR_MWh"].median()
    rest_mean_price = rest["price_EUR_MWh"].mean()
    rest_median_price = rest["price_EUR_MWh"].median()

    corr, pvalue = pearsonr(df["solar_MW"], df["price_EUR_MWh"])

    rows.append(
        {
            "country": cc,
            "strong_mean_price": strong_mean_price,
            "strong_median_price": strong_median_price,
            "rest_mean_price": rest_mean_price,
            "rest_median_price": rest_median_price,
            "corr_solar_price": corr,
            "pvalue": pvalue,
        }
    )

results = pd.DataFrame(rows)




5. Hypothesis Test: Merit-Order Effect of Solar Generation:

Results are printed per country, testing whether strong solar hours coincide with lower day-ahead prices — the expected merit-order effect. Two complementary signals are evaluated: a mean price comparison (strong vs. other hours) and a correlation threshold (|r| > 0.2) to distinguish meaningful from negligible solar-price relationships.

In [6]:
# Hypothesis test
print(
    "Hypothesis test: lower prices in strong solar hours due to high solar production\n"
)
for _, row in results.sort_values("country").iterrows():
    cc = row["country"]
    print(f"Country: {cc}")
    print(f"  Strong solar hours mean price:   {row['strong_mean_price']:.2f} EUR/MWh")
    print(
        f"  Strong solar hours median price: {row['strong_median_price']:.2f} EUR/MWh"
    )
    print(f"  Other hours mean price:          {row['rest_mean_price']:.2f} EUR/MWh")
    print(f"  Other hours median price:        {row['rest_median_price']:.2f} EUR/MWh")
    print(f"  Correlation(solar MW, price):    {row['corr_solar_price']:.3f}")
    print(f"  P-value:                         {row['pvalue']:.4f}")

    if row["strong_mean_price"] < row["rest_mean_price"]:
        print("  → Evidence that prices are lower on average in strong solar hours.")
    else:
        print(
            "  → No clear evidence that prices are lower in strong solar hours on average."
        )

    sig = row["pvalue"] < 0.05
    if row["corr_solar_price"] < -0.2 and sig:
        print(
            "  → Significant negative correlation: higher solar tends to suppress prices."
        )
    elif row["corr_solar_price"] < -0.2 and not sig:
        print("  → Negative correlation but not statistically significant.")
    elif row["corr_solar_price"] > 0.2 and sig:
        print(
            "  → Significant positive correlation: higher solar tends to raise prices."
        )
    elif row["corr_solar_price"] > 0.2 and not sig:
        print("  → Positive correlation but not statistically significant.")
    else:
        print("  → Solar output and price show only weak or insignificant correlation.")
    print()


Hypothesis test: lower prices in strong solar hours due to high solar production

Country: AT
  Strong solar hours mean price:   57.29 EUR/MWh
  Strong solar hours median price: 64.86 EUR/MWh
  Other hours mean price:          85.25 EUR/MWh
  Other hours median price:        84.95 EUR/MWh
  Correlation(solar MW, price):    -0.524
  P-value:                         0.0000
  → Evidence that prices are lower on average in strong solar hours.
  → Significant negative correlation: higher solar tends to suppress prices.

Country: BE
  Strong solar hours mean price:   45.06 EUR/MWh
  Strong solar hours median price: 47.45 EUR/MWh
  Other hours mean price:          74.84 EUR/MWh
  Other hours median price:        76.47 EUR/MWh
  Correlation(solar MW, price):    -0.522
  P-value:                         0.0000
  → Evidence that prices are lower on average in strong solar hours.
  → Significant negative correlation: higher solar tends to suppress prices.

Country: BG
  Strong solar hours mean pr

6. Welch's t-test — formal one-sided mean comparison

The Pearson correlation in block 5 tests a linear relationship between solar
generation (MW) and price (EUR/MWh). Welch's t-test complements it by formally
testing whether the mean price in strong-solar hours is statistically lower
than in rest hours. Welch (rather than pooled) is the correct choice because
the two groups have unequal sample sizes and unequal variances — evening peak
prices in rest hours are far more volatile than midday prices in strong-solar
hours, which would mis-calibrate a pooled t-test.

In [8]:
# Block 6: Welch's t-test — formal one-sided mean comparison.
#
# The Pearson correlation in block 5 tests a linear relationship between solar
# generation (MW) and price (EUR/MWh). Welch's t-test complements it by formally
# testing whether the mean price during strong-solar hours is LOWER than the
# mean price during rest hours.
#
# Welch (equal_var=False) is the right choice here because the two groups have
# unequal sample sizes AND unequal variances. We verify the variance inequality
# empirically per country before running the test (variance ratio rest/strong)
# rather than asserting it a priori.

from scipy.stats import ttest_ind

print("Welch's t-test — one-sided: strong-solar mean price < rest mean price\n")

welch_rows = []

for cc, df in country_dfs.items():
    strong = df[df["is_strong_solar_hour"]]["price_EUR_MWh"]
    rest = df[~df["is_strong_solar_hour"]]["price_EUR_MWh"]

    # Empirical variance check — justifies the choice of Welch over pooled Student's t.
    std_strong = strong.std()
    std_rest = rest.std()
    var_ratio = rest.var() / strong.var() if strong.var() > 0 else float("nan")

    t_stat, t_pvalue = ttest_ind(
        strong,
        rest,
        equal_var=False,  # Welch's t-test
        alternative="less",  # H1: mean(strong) < mean(rest)
    )

    welch_rows.append(
        {
            "country": cc,
            "n_strong": len(strong),
            "n_rest": len(rest),
            "std_strong": std_strong,
            "std_rest": std_rest,
            "var_ratio": var_ratio,
            "welch_t": t_stat,
            "welch_p": t_pvalue,
        }
    )

    pearson_p = float(results.loc[results["country"] == cc, "pvalue"].iloc[0])
    agree = (t_pvalue < 0.05) and (pearson_p < 0.05)

    print(f"Country: {cc}")
    print(f"  n strong / n rest:            {len(strong)} / {len(rest)}")
    print(f"  std strong / std rest:        {std_strong:.2f} / {std_rest:.2f} EUR/MWh")
    print(f"  variance ratio (rest/strong): {var_ratio:.2f}")
    print(f"  Welch t-statistic:            {t_stat:.3f}")
    print(f"  Welch p-value (one-sided):    {t_pvalue:.2e}")
    if t_pvalue < 0.05:
        print(
            "  → Statistically significant: strong-solar prices are lower on average."
        )
    else:
        print("  → Not significant at the 5% level.")
    if agree:
        print("  → Welch and Pearson agree on a significant solar-price relationship.")
    print()

welch_df = pd.DataFrame(welch_rows)
print("\nSummary:")
print(welch_df.to_string(index=False))


Welch's t-test — one-sided: strong-solar mean price < rest mean price

Country: AT
  n strong / n rest:            2354 / 2782
  std strong / std rest:        45.49 / 35.12 EUR/MWh
  variance ratio (rest/strong): 0.60
  Welch t-statistic:            -24.322
  Welch p-value (one-sided):    5.62e-123
  → Statistically significant: strong-solar prices are lower on average.
  → Welch and Pearson agree on a significant solar-price relationship.

Country: BE
  n strong / n rest:            2354 / 2782
  std strong / std rest:        42.88 / 33.05 EUR/MWh
  variance ratio (rest/strong): 0.59
  Welch t-statistic:            -27.484
  Welch p-value (one-sided):    7.52e-154
  → Statistically significant: strong-solar prices are lower on average.
  → Welch and Pearson agree on a significant solar-price relationship.

Country: BG
  n strong / n rest:            2354 / 2782
  std strong / std rest:        48.94 / 97.88 EUR/MWh
  variance ratio (rest/strong): 4.00
  Welch t-statistic:            -2